# RunPod ComfyUI All-in-One Generator

このノートブックは、RunPod環境で最速でComfyUIをセットアップし、バッチ生成からGoogle Driveへの自動転送、そして完了後の自動シャットダウンまでを自動化します。

## ワークフロー概要
1. **Cell 0 — API連携:** APIキーなどを入力（未設定時のみ）。秘密鍵のため取扱注意。
2. **Cell 1 — ワークフロー設定:** モデル選択・プロンプト・画像アスペクト比・LoRAなどのユーザー設定。
    3. **Cell 2 — 初期化:** 環境変数・パス・共通関数の初期化。
4. **Cell 3 — モデルDL:** `hf_transfer` で HuggingFace モデルを並列ダウンロード（目標10分以内）。
5. **Cell 4 — Drive設定 & LoRA DL:** RunPod SecretからGoogle Drive認証を復元し、自作LoRAを取得。アップローダー起動。
6. **Cell 5 — ComfyUI起動:** 最適化フラグを付けてバックグラウンド起動。起動完了をHTTPで確認してから次へ進む。
7. **Cell 6 — ワークフロー送信:** 選択したモデルとプロンプトでComfyUIにジョブをキューイング。
8. **Cell 7 — 自動監視 & 終了:** キュー空き + Drive同期完了でインスタンスを自動破棄。

## 事前準備（RunPod Secret登録）
| Secret名 | 内容 |
|---|---|
| `RCLONE_CONF_B64` | `rclone config` で生成した `rclone.conf` を `base64 -w0` でエンコードした文字列 |
| `MY_RUNPOD_API_KEY` | RunPod APIキー（自動停止に使用） |
| `HF_TOKEN` | HuggingFace Token（Publicモデルのみなら省略可） |

---

## Cell 0: APIキーの設定 (Credentials)
RunPodの「Secrets」機能で環境変数をあらかじめ設定している場合、このセルは**空欄のまま実行**して構いません。
Secretsを使用していない場合は、以下の `""` の中にご自身のキーを貼り付けてから実行してください。

※ ⚠️ **注意:** キーを入力した状態のノートブックを他人に共有・配布しないでください。

In [ ]:
# --------------------------------------------------
# ここにAPIキーを貼り付けます
# （RunPod Secrets で設定済みの場合は空欄でOKです）
# --------------------------------------------------
USER_HF_TOKEN        = ""
USER_RUNPOD_API_KEY  = ""
USER_RCLONE_CONF_B64 = ""
# --------------------------------------------------

import os

if USER_HF_TOKEN:
    os.environ['HF_TOKEN'] = USER_HF_TOKEN
if USER_RUNPOD_API_KEY:
    os.environ['MY_RUNPOD_API_KEY'] = USER_RUNPOD_API_KEY
if USER_RCLONE_CONF_B64:
    os.environ['RCLONE_CONF_B64'] = USER_RCLONE_CONF_B64

print("Cell 0: APIキーの設定を読み込みました。")

## Cell 1: ワークフロー設定 (User Configurations)

生成モデルや画角設定などを行います。

In [ ]:
import os
import subprocess
import time

def run(cmd):
    """コマンドを実行し、失敗時は例外を送出する"""
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, check=True)

# --- HuggingFace (RunPod Secret: HF_TOKEN) ---
HF_TOKEN = os.environ.get('HF_TOKEN', '')
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# --- RunPod API (RunPod Secret: MY_RUNPOD_API_KEY) ---
RUNPOD_API_KEY = os.environ.get('MY_RUNPOD_API_KEY', '')

# --- Google Drive (RunPod Secret: RCLONE_CONF_B64) ---
RCLONE_CONF_B64 = os.environ.get('RCLONE_CONF_B64', '')

# --- パス設定 ---
WORKSPACE    = '/workspace'
COMFYUI_DIR  = f'{WORKSPACE}/ComfyUI'
OUTPUT_DIR   = f'{COMFYUI_DIR}/output'
REMOTE_PATH  = 'gdrive:AI/ImageGeneration/RunPod_Output'

# プロンプトファイルの自動生成 (/workspace 直下)
PROMPT_FILE = f'{WORKSPACE}/runpod_prompts.txt'
if not os.path.exists(PROMPT_FILE):
    with open(PROMPT_FILE, 'w', encoding='utf-8') as f:
        f.write('# Inspire Pack形式でプロンプトを記述してください。\n')
        f.write('positive: A hyper-realistic portrait of a young woman,...\n')
        f.write('negative: worst quality, deformed,...\n')
        f.write('---\n')

# モデル選択
MODEL_NAME = 'zimage'  # 'turbo' または 'zimage'
WEIGHT_DTYPE = 'default'  # モデルの重み ('default', 'fp8_e4m3fn', 'fp16' 等)
BATCH_SIZE = 4  # 1つのプロンプトに対して同時に生成する枚数(バッチサイズ)

# ============================================================
# アスペクト比設定 (Aspect Ratio)
#   ▼ Instagram向けの主な選択肢 (以下の文字列から選んでコピペしてください)
#     '1:1 Square - 1080x1080'           (正方形)
#     'Instagram Portrait - 1080x1350'   (縦長 4:5)
#     'Instagram Story - 1080x1920'      (縦長 9:16 / ストーリーズ・リール用)
#     'Instagram Landscape - 1080x566'   (横長)
#   ▼ その他の例
#     '16:9 Cinema - 1920x1080', 'custom' などを指定可能
# ============================================================
ASPECT_RATIO = 'Instagram Portrait - 1080x1350'

# zimage等用 共通ネガティブプロンプト (turbo時は自動で無視されます)
DEFAULT_NEGATIVE = "worst quality, low quality, lowres, watermark, signature, deformed limbs, blurry face, distorted fingers, messy hair, cartoonish textures, overexposed highlights, asymmetrical eyes"

KSAMPLER_PARAMS = {
    'turbo': {
        'steps': 8,
        'cfg': 1.0,
        'sampler': 'dpmpp_2m',
        'scheduler': 'sgm_uniform',
    },
    'zimage': {
        'steps': 28,
        'cfg': 4.0,
        'sampler': 'dpmpp_2m',
        'scheduler': 'sgm_uniform',
    },
}



# LoRA設定
LORA_CONFIG = [
    {'name': 'zbase_jehyon_v1.safetensors', 'strength': 0.6},
    # {'name': 'hairdetailer.safetensors', 'strength': 0.2},  # 必要時はコメント解除
]

# --- LoRA の Google Drive パス ---
LORA_DRIVE_DIR = 'gdrive:AI/ImageGeneration/jehyon/LoRA'
LORA_FILENAMES = [
    'zbase_jehyon_v1.safetensors',
    'hairdetailer.safetensors',
]



## Cell 2: 環境初期化 & 共通関数 (Environment Setup)

In [ ]:
print('設定完了')
print(f'  MODEL_NAME  : {MODEL_NAME}')
print(f'  HF_TOKEN    : {"set" if HF_TOKEN else "not set (public models only)"}')
print(f'  RUNPOD_API  : {"set" if RUNPOD_API_KEY else "NOT SET - auto-shutdown disabled"}')
print(f'  RCLONE_CONF : {"set" if RCLONE_CONF_B64 else "NOT SET - Drive upload/LoRA DL disabled"}')


## Cell 3: HuggingFace モデル並列ダウンロード

In [ ]:
# NOTE: uv / hf_transfer はDockerイメージに焼き込み済みのため再インストール不要

print('Downloading models using parallel script...')
DOWNLOAD_SCRIPT = '/workspace/download_models.sh'

if os.path.exists(DOWNLOAD_SCRIPT):
    # download_models.sh は set -euo pipefail + PID追跡でエラー検知済み
    run(f'bash {DOWNLOAD_SCRIPT} {MODEL_NAME}')
else:
    # フォールバック: インラインで並列DL（終了コードを個別確認）
    print('Warning: download_models.sh not found, running inline download...')
    model_files = {
        'turbo': ('Comfy-Org/z_image_turbo', 'split_files/diffusion_models/z_image_turbo_bf16.safetensors'),
        'zimage': ('Comfy-Org/z_image', 'split_files/diffusion_models/z_image_bf16.safetensors'),
    }
    unet_repo, unet_file = model_files[MODEL_NAME]
    commands = [
        f'hf download {unet_repo} {unet_file} --local-dir {COMFYUI_DIR}/models/diffusion_models',
        f'hf download Comfy-Org/z_image_turbo split_files/vae/ae.safetensors --local-dir {COMFYUI_DIR}/models/vae',
        f'hf download Comfy-Org/z_image_turbo split_files/text_encoders/qwen_3_4b.safetensors --local-dir {COMFYUI_DIR}/models/text_encoders',
    ]
    procs = [(c, subprocess.Popen(c, shell=True)) for c in commands]
    results = [(c, p.wait()) for c, p in procs]
    errors = [(c, rc) for c, rc in results if rc != 0]
    if errors:
        for c, rc in errors:
            print(f'[ERROR] exit {rc}: {c}')
        raise RuntimeError('一部のモデルダウンロードが失敗しました。ログを確認してください。')

print('HFモデルダウンロード完了')


## Cell 4: rclone設定 & 認証テスト & LoRAダウンロード & アップローダー起動

In [ ]:
import base64

# --- rclone.conf 復元 ---
if not RCLONE_CONF_B64:
    raise RuntimeError(
        'RCLONE_CONF_B64 が未設定です。RunPod Secret に登録してください。\n'
        '生成方法: base64 -w0 ~/.config/rclone/rclone.conf'
    )

RCLONE_CONFIG_PATH = os.path.expanduser('~/.config/rclone/rclone.conf')
os.makedirs(os.path.dirname(RCLONE_CONFIG_PATH), exist_ok=True)
with open(RCLONE_CONFIG_PATH, 'w') as f:
    f.write(base64.b64decode(RCLONE_CONF_B64).decode('utf-8'))
print(f'rclone.conf を配置しました: {RCLONE_CONFIG_PATH}')

# --- rclone 認証テスト ---
print('\nGoogle Drive 接続テスト中...')
result = subprocess.run(
    ['rclone', 'lsd', 'gdrive:', '--max-depth', '1'],
    capture_output=True, text=True, timeout=30
)
if result.returncode != 0:
    raise RuntimeError(
        f'Google Drive 接続失敗 (code {result.returncode}):\n{result.stderr}\n'
        'OAuthトークンが失効している可能性があります。ローカルで rclone config reconnect gdrive: を実行してください。'
    )
print('Google Drive 認証OK')

# --- LoRA ダウンロード (Google Drive -> models/loras/) ---
LORA_DIR = f'{COMFYUI_DIR}/models/loras'
os.makedirs(LORA_DIR, exist_ok=True)
print(f'\nLoRAダウンロード中 ({len(LORA_FILENAMES)}ファイル)...')
for filename in LORA_FILENAMES:
    remote = f'{LORA_DRIVE_DIR}/{filename}'
    run(f'rclone copy "{remote}" "{LORA_DIR}" --progress')
    print(f'  {filename} -> {LORA_DIR}')
print('LoRAダウンロード完了')

# --- Google Drive アップローダー起動 ---
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('\nAuto Uploader を起動中...')
subprocess.Popen(
    ['python', '/workspace/scripts/auto_uploader.py',
     '--watch_dir', OUTPUT_DIR,
     '--remote_path', REMOTE_PATH,
     '--interval', '30'],
    stdout=open('/workspace/uploader.log', 'w'),
    stderr=subprocess.STDOUT,
)
print('アップローダー起動完了 (ログ: /workspace/uploader.log)')

## Cell 5: ComfyUI 起動 (APIモード/最適化) & 起動確認

In [ ]:
import urllib.request
import urllib.error

print('GPUリソースモニタリングを開始中...')
subprocess.Popen(
    f'nvidia-smi --query-gpu=timestamp,utilization.gpu,memory.used,memory.total --format=csv -l 5 > {OUTPUT_DIR}/gpu_usage_log.csv',
    shell=True
)

print('ComfyUI を起動中...')
LAUNCH_CMD = [
    'python', f'{COMFYUI_DIR}/main.py',
    '--listen', '0.0.0.0',
    '--highvram',
    '--bf16-vae',
    '--fast',
    '--preview-method', 'none',
    '--output-directory', OUTPUT_DIR,
]
subprocess.Popen(
    LAUNCH_CMD,
    stdout=open('/workspace/comfyui.log', 'w'),
    stderr=subprocess.STDOUT,
)

# --- 起動確認: HTTP応答が返るまで最大3分ポーリング ---
print('起動確認中 (最大3分)...')
deadline = time.time() + 180
while time.time() < deadline:
    try:
        urllib.request.urlopen('http://localhost:8188', timeout=3)
        print('ComfyUI 起動確認OK')
        print('APIエンドポイント: http://localhost:8188')
        break
    except (urllib.error.URLError, OSError):
        time.sleep(5)
else:
    raise RuntimeError(
        'ComfyUI が3分以内に起動しませんでした。\n'
        '詳細は /workspace/comfyui.log を確認してください。'
    )

## Cell 6: ワークフロー送信 (Workflow Submission)
選択したモデルとプロンプトでComfyUIにジョブをキューイングします。

In [ ]:
import json
import random
import urllib.request


def build_workflow(prompt_text, negative_prompt, model_name, aspect_ratio, weight_dtype, ksampler_params, lora_config, batch_size=2, seed=None):
    if seed is None:
        seed = random.randint(0, 2**53)

    model_files = {
        'turbo': 'split_files/diffusion_models/z_image_turbo_bf16.safetensors',
        'zimage': 'split_files/diffusion_models/z_image_bf16.safetensors',
    }

    api = {
        '1': {'class_type': 'UNETLoader', 'inputs': {
            'unet_name': model_files[model_name],
            'weight_dtype': weight_dtype,
        }},
        '2': {'class_type': 'VAELoader', 'inputs': {
            'vae_name': 'split_files/vae/ae.safetensors',
        }},
        '3': {'class_type': 'CLIPLoader', 'inputs': {
            'clip_name': 'split_files/text_encoders/qwen_3_4b.safetensors',
            'type': 'lumina2',
            'device': 'default',
        }},
        '8': {'class_type': 'CR Aspect Ratio Social Media', 'inputs': {
            'width': 1024,
            'height': 1024,
            'aspect_ratio': aspect_ratio,
            'swap_dimensions': 'Off',
            'upscale_factor': 1,
            'prescale_factor': 1,
            'batch_size': batch_size,
        }},
    }

    # LoRAチェーンを構築
    current_model = ['1', 0]
    current_clip = ['3', 0]
    for i, lora in enumerate(lora_config):
        lora_id = f'lora_{i}'
        api[lora_id] = {
            'class_type': 'LoraLoader',
            'inputs': {
                'model': current_model,
                'clip': current_clip,
                'lora_name': lora['name'],
                'strength_model': lora['strength'],
                'strength_clip': lora['strength'],
            }
        }
        current_model = [lora_id, 0]
        current_clip = [lora_id, 1]

    api['5'] = {'class_type': 'ModelSamplingAuraFlow', 'inputs': {
        'model': current_model,
        'shift': 3.0,
    }}
    api['6'] = {'class_type': 'CLIPTextEncode', 'inputs': {
        'clip': current_clip,
        'text': prompt_text,
    }}
    if model_name == 'turbo':
        api['7'] = {'class_type': 'ConditioningZeroOut', 'inputs': {'conditioning': ['6', 0]}}
    else:
        api['7'] = {'class_type': 'CLIPTextEncode', 'inputs': {'clip': current_clip, 'text': negative_prompt}}
    api['9'] = {'class_type': 'KSampler', 'inputs': {
        'model': ['5', 0],
        'positive': ['6', 0],
        'negative': ['7', 0],
        'latent_image': ['8', 5],
        'seed': seed,
        'steps': ksampler_params['steps'],
        'cfg': ksampler_params['cfg'],
        'sampler_name': ksampler_params['sampler'],
        'scheduler': ksampler_params['scheduler'],
        'denoise': 1.0,
    }}
    api['10'] = {'class_type': 'VAEDecodeTiled', 'inputs': {
        'samples': ['9', 0],
        'vae': ['2', 0],
        'tile_size': 512,
        'overlap': 64,
        'temporal_size': 64,
        'temporal_overlap': 8,
    }}
    api['11'] = {'class_type': 'SaveImage', 'inputs': {
        'images': ['10', 0],
        'filename_prefix': f'RunPod_{model_name}',
    }}
    return api


# プロンプトを読み込む
if PROMPT_FILE and os.path.exists(PROMPT_FILE):
    with open(PROMPT_FILE, 'r', encoding='utf-8') as f:
        content = f.read()
    blocks = [b.strip() for b in content.split('---') if b.strip()]
    prompts = []
    for block in blocks:
        pos = ''
        neg = DEFAULT_NEGATIVE
        for l in block.split('\n'):
            if l.lower().startswith('positive:'):
                pos = l[9:].strip()
            elif l.lower().startswith('negative:'):
                neg = l[9:].strip()
        if pos:
            prompts.append({'positive': pos, 'negative': neg})
    print(f'プロンプトファイル: {PROMPT_FILE} ({len(prompts)}件)')
else:
    prompts = []

if not prompts:
    raise ValueError('プロンプトが未設定です (PROMPT_FILE を Cell 1 で設定してください)')

COMFY_URL = 'http://localhost:8188'
print(f'\nモデル: {MODEL_NAME} | {len(prompts)}件をキューに追加中...')

for i, p_dict in enumerate(prompts):
    if isinstance(p_dict, str): p_dict = {'positive': p_dict, 'negative': DEFAULT_NEGATIVE}
    workflow = build_workflow(
        prompt_text=p_dict['positive'],
        negative_prompt=p_dict['negative'],
        model_name=MODEL_NAME,
        aspect_ratio=ASPECT_RATIO,
        weight_dtype=WEIGHT_DTYPE,
        ksampler_params=KSAMPLER_PARAMS[MODEL_NAME],
        lora_config=LORA_CONFIG,
        batch_size=BATCH_SIZE,
    )
    payload = json.dumps({'prompt': workflow}).encode('utf-8')
    req = urllib.request.Request(
        f'{COMFY_URL}/prompt',
        data=payload,
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    with urllib.request.urlopen(req) as resp:
        result = json.loads(resp.read())
        pid = result['prompt_id']
    print(f'  [{i+1}/{len(prompts)}] batch({BATCH_SIZE}) queued: {pid}')

print(f'\n全 {len(prompts)} 件のワークフローをキューに追加しました。')


## Cell 7: 自動監視 & 終了 (Auto-Shutdown)

In [ ]:
print('Auto-Terminator を起動中...')
subprocess.Popen(
    ['python', '/workspace/scripts/auto_terminator.py',
     '--timeout_mins', '60',
     '--idle_mins', '1'],
    stdout=open('/workspace/terminator.log', 'w'),
    stderr=subprocess.STDOUT,
)
print('自動停止スクリプトを有効化しました (ログ: /workspace/terminator.log)')
print('キュー空き + Drive同期完了でインスタンスを自動破棄します')